# Materials-Triage — deterministic pipeline demo (LIVE)

This notebook runs the **deterministic core** end to end against the **live Materials Project REST API** —
with **no LLM** anywhere. It exercises the real pipeline stages:

> `MaterialsProjectAdapter.retrieve(spec)` → `apply_hard_filters` → `rank` → `TriageResult`

Every number comes from the database and carries its `Provenance`; deterministic code does all filtering and
ranking. The LLM layer (spec-building from language, grounded narrative) is deliberately *not* part of this
slice — though we first show the **schema-derived vocabulary** that layer would be constrained to.

**Scenario:** find a **Zn–O** compound that is a semiconductor (band gap **1.0–4.0 eV**), preferring a
**wide gap** and **low density**.

## Prerequisites

1. Install the package (pulls `requests`):  `pip install -e .`
2. Export your key:  `export X_API_KEY=...`  (32-char sandbox key)

> Note: the sandbox sits behind Cloudflare, which **bans the default `Python-urllib` User-Agent** (HTTP 403,
> error 1010). `requests` sends a UA that passes, which is why the adapter uses it.

Results are **live**, so exact numbers and ids will drift as the database updates.

In [ ]:
# Optional: loads X_API_KEY from a local .env. Or install the extra: pip install -e ".[notebook]"
!pip install python-dotenv

In [ ]:
# Make the package importable whether the kernel starts in the repo root or in notebooks/.
import os
import pathlib
import sys
from collections import Counter

try:
    import materials_triage  # noqa: F401
except ModuleNotFoundError:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "materials_triage").exists():
            sys.path.insert(0, str(base / "src"))
            break

from dotenv import load_dotenv

from materials_triage.core.ranking import rank
from materials_triage.core.schema import Constraint, RankingTarget, TriageSpec
from materials_triage.core.scoring import apply_hard_filters
from materials_triage.sources.materials_project import MaterialsProjectAdapter

load_dotenv()

if not os.environ.get("X_API_KEY"):
    print("WARNING: X_API_KEY is not set — the live cells will return HTTP 403.")


def fmt_pv(pv):
    """Render a PropertyValue for display (no unit shown for a dimensionless value)."""
    if pv is None:
        return "absent"
    if pv.missing:
        return "missing"
    unit = f" {pv.unit}" if pv.unit else ""  # unit is None for dimensionless quantities
    return f"{pv.value:.4g}{unit}"

## The source's property vocabulary (schema-derived)

Before any request, the adapter publishes **which property names it can actually return** — and their units.
This surface is **derived from the Materials Project OpenAPI schema** and committed as a static table
(`tools/gen_mp_vocab.py` → `_mp_fields.py`); `property_vocabulary()` reads it with **no live fetch**, so runs
stay replayable. In the full product this is exactly what the spec-builder / LLM is constrained to, so a
request can only name properties `retrieve` will populate — **no silent empty results** from a hallucinated
field name. A unit of `None` (shown as *no unit*) means the value has no physical unit — a **dimensionless**
quantity (refractive index, dielectric constant), a **boolean** flag (`is_metal`), or a bare **count**
(`nelements`).

In [ ]:
# property_vocabulary() needs no network — it returns the committed, schema-derived table.
vocab = MaterialsProjectAdapter().property_vocabulary()

print(f"{len(vocab)} retrievable properties (schema-derived):\n")
for name, unit in sorted(vocab.items()):
    print(f"  {name:42} {unit if unit is not None else '(no unit)'}")

## 1. The request → a `TriageSpec`

In the full product the LLM builds this from natural language. Here we write it directly: a **composition**
requirement (Zn + O), one **hard constraint** (the band-gap window), and two **ranking targets** whose weights
are proportional shares summing to 1.

In [ ]:
spec = TriageSpec(
    constraints=(Constraint(property_name="band_gap", min=1.0, max=4.0),),
    ranking_targets=(
        RankingTarget(property_name="band_gap", direction="maximize", weight=0.6),
        RankingTarget(property_name="density", direction="minimize", weight=0.4),
    ),
    required_elements=frozenset({"Zn", "O"}),
)
spec

## 2. Retrieve — LIVE

`MaterialsProjectAdapter()` with no injected transport hits the real API (reading `X_API_KEY`). The adapter
scopes the query from the spec: it pushes the **composition** (`elements=O,Zn`) server-side and requests only
the `_fields` the run will read — but it leaves the **numeric** filtering to the deterministic core. Note the
anonymized `mp-aaa…` ids: the mirror remaps every returned id, and the adapter stores the *returned* one.

In [ ]:
adapter = MaterialsProjectAdapter()  # real transport; offline tests inject a fake instead
candidates = adapter.retrieve(spec)

print(f"retrieved {len(candidates)} Zn-O materials (showing 5):")
for c in candidates[:5]:
    gap = fmt_pv(c.properties.get("band_gap"))
    density = fmt_pv(c.properties.get("density"))
    print(f"  {c.identifier:13} {c.formula:16} band_gap={gap:12} density={density}")

## 3. Hard filters

`apply_hard_filters` drops every candidate outside the band-gap window, recording a **structured reason** per
drop (never a silent disappearance) — metallic Zn-O phases fall below the min, wide-gap insulators above the
max.

In [ ]:
survivors, excluded = apply_hard_filters(candidates, spec.constraints)

print(f"{len(survivors)} survive, {len(excluded)} excluded")
print("exclusion reasons:", dict(Counter(e.reason for e in excluded)))
print("\na few exclusions:")
for e in excluded[:4]:
    print(
        f"  {e.candidate.formula:16} {e.property_name} {e.reason} "
        f"(value {e.value:.3g} vs bound {e.bound})"
    )

## 4. Ranking (weighted average)

`rank` normalizes each target onto [0, 1] in its direction, multiplies by the proportional weight, and sums.
`contributions` shows each target's weighted share (they sum to the score). Any candidate missing a ranked
property is imputed a neutral 0.5 and **flagged** — the gap is never silently credited.

In [ ]:
result = rank(survivors, spec.ranking_targets)

print(f"top 10 of {len(result.ranked)} ranked:\n")
for i, sc in enumerate(result.ranked[:10], 1):
    contribs = ", ".join(f"{k}:{v:+.3f}" for k, v in sorted(sc.contributions.items()))
    flags = f"   imputed -> {sorted(sc.flagged_missing)}" if sc.flagged_missing else ""
    print(
        f"{i:2}. {sc.candidate.formula:16} {sc.candidate.identifier:13} "
        f"score={sc.score:.3f}  ({contribs}){flags}"
    )

## 5. Everything is provenance-tagged

Each number records where it came from — this is what lets the (future) output validator reject any value the
LLM did not get from a real source.

In [ ]:
top = result.ranked[0]
print(f"winner: {top.candidate.formula} ({top.candidate.identifier})  score={top.score:.3f}\n")
for name, pv in top.candidate.properties.items():
    p = pv.provenance
    print(f"  {name:10} = {fmt_pv(pv):14} <- {p.source} / {p.record_id}")

## What a representative run looks like

Live numbers drift, but a recent run produced roughly:

```
retrieved 100 Zn-O materials; 39 survive band_gap[1,4]; 61 excluded
exclusion reasons: {'below_min': 60, 'above_max': 1}

TOP RANKED:
  1. Sc2ZnO4     mp-aaaieiso   score=0.947  (band_gap:+0.600, density:+0.347)
  2. ZnCoP2O7    mp-aaadpghk   score=0.843  (band_gap:+0.463, density:+0.380)
  ...
```

**Takeaways:** the pipeline never lets the LLM invent a number; deterministic code filters and ranks; missing
data is ranked-but-flagged rather than dropped or guessed; every value is traceable to its source. The exact
same code runs **offline** in tests by injecting a fake transport —
`MaterialsProjectAdapter(http_get=lambda url, params, headers: FIXTURE)` — which is how the suite stays
deterministic without the network.